#### Set up

In [1]:
!uv add deepagents langchain langchain-openai langgraph langgraph-checkpoint-postgres "psycopg[binary,pool]" tavily-python langfuse pydantic python-dotenv mermaid-magic

Resolved 110 packages in 13ms
Checked 96 packages in 19ms


In [2]:
import json
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

import mermaid_magic   # activates Mermaid rendering (the %%mermaid magic); show_harness uses it to draw agent graphs

load_dotenv(find_dotenv(usecwd=True), override=True)

for key in ["OPENROUTER_API_KEY", "TAVILY_API_KEY", "DATABASE_URL"]:
    assert os.environ.get(key), f"Set {key} in .env"

MODEL = ChatOpenAI(
    model=os.environ.get("OPENROUTER_MODEL", "nvidia/nemotron-3.5-lightning:free"),
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
    # OpenRouter model names have no built-in profile. Without one, Deep Agents falls back to a 170k-token
    # summarization trigger, which is above this model's 128k context window.
    profile={"max_input_tokens": 128000},
)

# Agent files live here; the agent sees this folder as "/"
WORKSPACE = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "workspace"
print("Model:", MODEL.model_name)
print("Workspace:", WORKSPACE)

Model: nvidia/nemotron-3.5-lightning:free
Workspace: /Users/syedirizvi/multi-agent/workspace


#### LangFuse Setup

In [3]:
from langfuse import get_client
from langfuse.langchain import CallbackHandler

langfuse = get_client()
print("Langfuse auth OK:", langfuse.auth_check())
langfuse_handler = CallbackHandler()

import time
NOTEBOOK_SESSION = f"research-desk-notebook-{time.strftime('%m%d-%H%M')}"   # groups every run from this kernel


def run_config(name: str, thread_id: str | None = None, session_id: str | None = None) -> dict:
    """Config for one run: Langfuse tracing, trace name, session, optional thread id.
    """
    cfg = {
        "callbacks": [langfuse_handler],
        "run_name": name,
        "metadata": {
            "langfuse_session_id": session_id or NOTEBOOK_SESSION,
            "langfuse_tags": ["research-desk"],
        },
    }
    if thread_id:
        cfg["configurable"] = {"thread_id": thread_id}
    return cfg


Langfuse auth OK: True


#### Web search and page extraction tools

In [4]:
from langchain_core.tools import tool
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
MAX_PAGE_CHARS = 6000


def format_result(r: dict) -> dict:
    return {"title": r.get("title"), "url": r.get("url"), "snippet": r.get("content")}


@tool
def search_web(query: str, max_results: int = 5) -> str:
    """Search the web for credible sources. Returns title, url and a snippet per result."""
    try:
        res = tavily.search(query=query, max_results=max_results)
    except Exception as e:
        return json.dumps({"error": f"search failed: {e}"})
    return json.dumps([format_result(r) for r in res.get("results", [])])


@tool
def extract_pages(urls: list[str]) -> str:
    """Extract readable text from up to 5 URLs. Text is truncated per page."""
    try:
        res = tavily.extract(urls=urls[:5])
    except Exception as e:
        return json.dumps({"error": f"extract failed: {e}"})
    return json.dumps({
        "pages": [{"url": r["url"], "text": r["raw_content"][:MAX_PAGE_CHARS]} for r in res.get("results", [])],
        "failed": [f.get("url") for f in res.get("failed_results", [])],
    })

In [5]:
out = search_web.invoke({"query": "AI coding agents impact on software jobs in US", "max_results": 3})
for r in json.loads(out):
    print(r["title"], "|", r["url"])
print("\nChars returned by one search:", len(out))

AI Coding Agents Job Impact Analysis 2026 | https://www.themodernblog.com/ai-coding-agents-job-impact-analysis-2026
Coding Jobs and AI in 2026: Copilot Adoption, Junior Dev Displacement, and New Roles | AIExposure | https://www.aiexposure.org/analysis/coding-jobs-ai-2026
AI impact on software developer jobs (2026 data) | ShipFastDev | https://shipfastdev.com/blog/ai-impact-on-software-developer-jobs

Chars returned by one search: 4177


#### Basic DeepAgent Agent

In [6]:
from deepagents import create_deep_agent

first_agent = create_deep_agent(
    name="first-agent",
    model=MODEL,
    tools=[search_web, extract_pages],
    system_prompt="""You are a research manager.
Investigate the user's request, gather credible evidence,
and produce a structured report with citations.""",
)

#### Utility Methods

In [7]:
def _short(x, n=150):
    s = " ".join(str(x).split())
    return s if len(s) <= n else s[:n] + "…"

ICON = {"completed": "✅", "in_progress": "🔄", "pending": "⬜"}


def print_message(m, who="manager", step=None, result_chars=150, answer_chars=2500):
    """Print one message as a single readable line (or block for the final answer)."""
    tag = f"[{who}] " if who != "manager" else ""
    if m.type == "human":
        print(f"👤 USER: {_short(m.content, 300)}\n")
    elif m.type == "ai":
        if m.tool_calls:
            for c in m.tool_calls:
                if c["name"] == "write_todos":
                    print(f"📝 {tag}write_todos")
                    for t in c["args"]["todos"]:
                        print(f"     {ICON.get(t['status'], '•')} {_short(t['content'], 100)}")
                elif c["name"] == "read_file" and str(c["args"].get("file_path", "")).startswith("/skills/"):
                    print(f"📖 {tag}loaded skill: {c['args']['file_path']}")
                else:
                    print(f"🔧 {tag}{c['name']}  {_short(json.dumps(c['args']), 130)}")
        elif m.content:
            print(f"\n🤖 {tag}ANSWER\n{'-' * 60}\n{str(m.content)[:answer_chars]}")
    elif m.type == "tool" and m.name != "write_todos":
        print(f"   ↳ {m.name}: {_short(m.content, result_chars)}  ({len(str(m.content)):,} chars)")


def show_trace(result, answer_chars=2500):
    """Readable timeline of a finished run, plus token and cost totals."""
    tokens = cost = model_calls = 0
    for m in result["messages"]:
        print_message(m, answer_chars=answer_chars)
        if m.type == "ai":
            usage = m.response_metadata.get("token_usage", {})
            tokens += usage.get("total_tokens", 0)
            cost += usage.get("cost") or 0
            model_calls += 1
    if result.get("todos"):
        print("\n📋 Final todos")
        for t in result["todos"]:
            print(f"   {ICON.get(t['status'], '•')} {t['content']}")
    if result.get("files"):
        print("\n📁 Files in state:", list(result["files"]))
    print(f"\n— {model_calls} model calls · {tokens:,} tokens · ${cost:.4f} (manager only)")


def stream_events(agent, payload, config=None, answer_chars=1500):
    """Stream a run and print each step as it happens, including subagent steps. Returns any interrupts."""
    seen, interrupts = set(), []
    for namespace, update in agent.stream(
        payload, config=config or run_config("live-run"), stream_mode="updates", subgraphs=True,
    ):
        who = "manager" if not namespace else "subagent"
        for node, content in update.items():
            if node == "__interrupt__":
                interrupts.extend(content)
                continue
            if not isinstance(content, dict):
                continue
            msgs = content.get("messages")
            msgs = getattr(msgs, "value", msgs)   # unwrap Overwrite
            for m in msgs or []:
                if getattr(m, "id", None) in seen:
                    continue
                seen.add(getattr(m, "id", None))
                print_message(m, who=who, answer_chars=answer_chars if who == "manager" else 400)
    return list({getattr(i, "id", id(i)): i for i in interrupts}.values())


def run_live(agent, text, config=None):
    return stream_events(agent, {"messages": [{"role": "user", "content": text}]}, config)


_PREVIOUS_HARNESS = {"hooks": set(), "tools": set()}


def draw_harness(graph) -> str:
    """Text diagram of the agent loop: which middleware hooks run before/after the model."""
    edges = {}
    for e in graph.edges:
        edges.setdefault(e.source, []).append(e.target)
    order, seen, queue = [], {"__start__"}, ["__start__"]
    while queue:
        for target in edges.get(queue.pop(0), []):
            if target not in seen:
                seen.add(target)
                order.append(target)
                queue.append(target)

    def hooks(kind):
        return [n.split(".")[0].replace("Middleware", "").lstrip("_") for n in order if n.endswith("." + kind)]

    lines = ["start"]
    if hooks("before_agent"):
        lines.append("  → before_agent (once):  " + " → ".join(hooks("before_agent")))
    if hooks("before_model"):
        lines.append("  → before_model:         " + " → ".join(hooks("before_model")))
    lines.append("  → MODEL  (the LLM decides: answer, or call tools)")
    if hooks("after_model"):
        lines.append("  → after_model:          " + " → ".join(hooks("after_model")))
    lines.append("  → tool calls?  yes → TOOLS → back to the start of the loop")
    after = " → ".join(hooks("after_agent") + ["end"])
    lines.append(f"                 no  → {after}")
    return "\n".join(lines)


def show_graph(graph):
    """Render the agent graph with Mermaid. If it cannot render, fall back to a plain arrow diagram."""
    try:
        mermaid_magic.mermaid("", graph.draw_mermaid())
    except Exception as error:
        print(f"(Mermaid unavailable: {type(error).__name__}. Plain diagram instead; or paste agent.get_graph().draw_mermaid() into https://mermaid.live)")
        print("\nAgent loop:\n" + draw_harness(graph))


def show_harness(agent, title: str, subagents=None, draw: bool = True, compare: bool = True):
    """Print the harness of a Deep Agent: hook nodes, tools, subagents, and what is new since the last one shown."""
    graph = agent.get_graph()
    hook_nodes = [n for n in graph.nodes if n not in ("__start__", "__end__", "model", "tools")]
    tools = list(agent.nodes["tools"].bound.tools_by_name)
    print(f"🧩 HARNESS: {title}\n{'=' * 64}")
    print("Hook nodes (middleware that runs in the loop):")
    for n in hook_nodes:
        print("   •", n)
    print(f"Tools ({len(tools)}):", ", ".join(tools))
    if subagents:
        print("Subagents:")
        for s in subagents:
            names = [getattr(t, "name", str(t)) for t in s.get("tools", [])]
            extras = []
            if s.get("skills"):
                extras.append("skills=" + ",".join(s["skills"]))
            if s.get("permissions"):
                extras.append(f"{len(s['permissions'])} permission rules")
            print(f"   • {s['name']}: tools={names}" + (f"  [{'; '.join(extras)}]" if extras else ""))
    if compare and (_PREVIOUS_HARNESS["hooks"] or _PREVIOUS_HARNESS["tools"]):
        new_hooks = sorted(set(hook_nodes) - _PREVIOUS_HARNESS["hooks"])
        new_tools = sorted(set(tools) - _PREVIOUS_HARNESS["tools"])
        print("➕ New since the previous harness shown:",
              (new_hooks + new_tools) or "(nothing; the difference is in prompts, backend or config)")
    _PREVIOUS_HARNESS.update(hooks=set(hook_nodes), tools=set(tools))
    if draw:
        show_graph(graph)

In [8]:
show_harness(first_agent, "1 · first agent: just tools and defaults")

🧩 HARNESS: 1 · first agent: just tools and defaults
Hook nodes (middleware that runs in the loop):
   • _ParentNotifierMiddleware.after_agent
   • PatchToolCallsMiddleware.before_agent
Tools (11): ls, read_file, write_file, edit_file, delete, glob, grep, execute, task, search_web, extract_pages


In [9]:
DEFAULT_BRIEF = (
    "Research the impact of AI coding agents on software engineering jobs in US. "
    "Cover hiring trends, salary impact, company adoption, and credible supporting evidence."
)
BRIEF = input("Research brief (press Enter for the default): ").strip() or DEFAULT_BRIEF
print("Brief:", BRIEF)

result = first_agent.invoke({"messages": [{"role": "user", "content": BRIEF}]}, config=run_config("first-agent"))
show_trace(result)

Brief: Research the impact of AI coding agents on software engineering jobs in US. Cover hiring trends, salary impact, company adoption, and credible supporting evidence.
👤 USER: Research the impact of AI coding agents on software engineering jobs in US. Cover hiring trends, salary impact, company adoption, and credible supporting evidence.

🔧 search_web  {"query": "AI coding agents impact software engineering jobs US hiring trends 2024", "max_results": 5}
   ↳ search_web: [{"title": "Andrew Ng on X: \"As AI agents accelerate coding, what is the future of software engineering? Some trends are clear, such as the Product M…  (8,113 chars)
🔧 search_web  {"query": "AI coding agents salary impact software engineers US wage premium 2024 2025", "max_results": 5}
   ↳ search_web: [{"title": "Average software engineer salary in 2026", "url": "https://eltexsoft.com/blog/software-developer-salary-statistics", "snippet": "Yes, acro…  (6,817 chars)
🔧 search_web  {"query": "AI coding agents company 

#### Subtasks with todos

In [10]:
from langchain.agents.middleware import TodoListMiddleware
from langchain.agents.middleware.todo import WRITE_TODOS_SYSTEM_PROMPT, WRITE_TODOS_TOOL_DESCRIPTION

TODO_RULES = """

## Keeping the list reliable

- Every `write_todos` call must contain the COMPLETE list: keep finished items and change only their status.
- Do not remove completed items; the list is the record of the work.
- Mark an item `in_progress` before starting it and `completed` as soon as it is done.
- Before your final answer, every item must be `completed`."""

# The default description tells the model to delete finished tasks; keep them instead.
_old = "Remove tasks that are no longer relevant from the list entirely"
assert _old in WRITE_TODOS_TOOL_DESCRIPTION
TODO_TOOL_DESCRIPTION = WRITE_TODOS_TOOL_DESCRIPTION.replace(
    _old, "Keep completed tasks in the list; only remove tasks that turned out to be irrelevant")


def make_todo_middleware() -> TodoListMiddleware:
    return TodoListMiddleware(system_prompt=WRITE_TODOS_SYSTEM_PROMPT + TODO_RULES, tool_description=TODO_TOOL_DESCRIPTION)

In [11]:
planning_agent = create_deep_agent(
    name="planning-agent",
    model=MODEL,
    tools=[search_web, extract_pages],
    middleware=[make_todo_middleware()],
    system_prompt="""You are a research manager.
First write a todo list that splits the request into independent research tasks.
Update todo status as you progress. Finish with a short cited summary.""",
)

In [12]:
BRIEF = (
    "Research the impact of AI coding agents on software engineering jobs in US. "
    "Cover hiring trends, salary impact, company adoption, and credible supporting evidence."
)
run_live(planning_agent, BRIEF)

📝 write_todos
     🔄 Research hiring trends for software engineering jobs in US with AI coding agents
     ⬜ Research salary impact of AI coding agents on software engineering roles in US
     ⬜ Research company adoption of AI coding agents in software engineering
     ⬜ Synthesize findings into a short cited summary covering all areas
🔧 search_web  {"query": "AI coding agents impact software engineering hiring trends US 2024", "max_results": 10}
   ↳ search_web: [{"title": "Andrew Ng on X: \"As AI agents accelerate coding, what is the future of software engineering? Some trends are clear, such as the Product M…  (15,039 chars)
🔧 search_web  {"query": "AI coding agents salary impact software engineers US 2024 2025 survey", "max_results": 10}
   ↳ search_web: [{"title": "AI agents will replace your $1M+ engineering team | Christian Reyes posted on the topic | LinkedIn", "url": "https://www.linkedin.com/post…  (13,996 chars)
🔧 search_web  {"query": "AI coding agents company adoption soft

[]

#### SubAgents Delegation

In [16]:
SCHOLARLY_DOMAINS = [
    "arxiv.org", "ssrn.com", "nber.org", "semanticscholar.org", "doj.org",
    "sciencedirect.com", "springer.com", "nature.com", "acm.org", "ieee.org",
    "researchgate.net", "repec.org", "oecd.org", "worldbank.org", "imf.org",
]

@tool
def search_scholarly(query: str, max_results: int = 5) -> str:
    """Search scholarly and institutional research (papers, working papers, reports)."""
    try:
        res = tavily.search(query=query, max_results=max_results, include_domains=SCHOLARLY_DOMAINS)
    except Exception as e:
        return json.dumps({"error": f"search failed: {e}"})
    return json.dumps([format_result(r) for r in res.get("results", [])])

In [17]:
WEB_RESEARCH_PROMPT = """You are a web researcher.
Find credible web sources for the topic you are given and extract the most useful pages.
Save detailed evidence to files (paths are given in your task). Keep your reply short:
main findings, limitations, source count, and the saved file paths.
Do not paste raw search output into your reply.
Prefer sources specific to the country, region or population named in your task. If only global or other-region
data exists, use it only as context and label it "global" or "not region-specific" in your files and reply.
Never present global figures as if they described the requested region."""

ACADEMIC_RESEARCH_PROMPT = """You are an academic researcher.
Find scholarly papers, working papers and institutional studies on the topic you are given.
Label each as peer-reviewed, preprint or institutional report. Save detailed evidence to files
(paths are given in your task). Reply with main findings, evidence strength, source count and file paths.
Prefer sources specific to the country, region or population named in your task. If only global or other-region
data exists, use it only as context and label it "global" or "not region-specific" in your files and reply.
Never present global figures as if they described the requested region."""

research_subagents = [
    {
        "name": "web_researcher",
        "description": "Research a topic using credible web sources: news, company reports, industry data.",
        "system_prompt": WEB_RESEARCH_PROMPT,
        "tools": [search_web, extract_pages],
    },
    {
        "name": "academic_researcher",
        "description": "Find and summarize scholarly research, working papers and institutional studies.",
        "system_prompt": ACADEMIC_RESEARCH_PROMPT,
        "tools": [search_scholarly],
    },
]

In [18]:
RESEARCH_MANAGER_PROMPT = """You are the Research Manager of a research desk.

Workflow:
1. Write a todo list that splits the brief into independent research dimensions.
2. Delegate web topics to `web_researcher` and scholarly topics to `academic_researcher` with the `task` tool.
   You may start independent delegations in the same step.
3. Read the saved findings, then write a draft report.
4. Never do heavy searching yourself; specialists do that.

Todo discipline: send the full list on every write_todos call, change only statuses, and mark all items completed before your final answer.
Always cite sources. Do not invent facts or URLs."""


delegating_agent = create_deep_agent(
    name="research-manager",
    model=MODEL,
    middleware=[make_todo_middleware()],
    subagents=research_subagents,
    system_prompt=RESEARCH_MANAGER_PROMPT,
)

show_harness(delegating_agent, "3 · + specialist subagents", subagents=research_subagents)

result = delegating_agent.invoke({"messages": [{"role": "user", "content": BRIEF + " Keep it to two dimensions."}]}, config=run_config("delegating-agent"))
show_trace(result)

🧩 HARNESS: 3 · + specialist subagents
Hook nodes (middleware that runs in the loop):
   • _ParentNotifierMiddleware.after_agent
   • PatchToolCallsMiddleware.before_agent
   • TodoListMiddleware.after_model
Tools (10): ls, read_file, write_file, edit_file, delete, glob, grep, execute, task, write_todos
Subagents:
   • web_researcher: tools=['search_web', 'extract_pages']
   • academic_researcher: tools=['search_scholarly']
➕ New since the previous harness shown: (nothing; the difference is in prompts, backend or config)


👤 USER: Research the impact of AI coding agents on software engineering jobs in US. Cover hiring trends, salary impact, company adoption, and credible supporting evidence. Keep it to two dimensions.

📝 write_todos
     🔄 Research hiring trends and salary impact of AI coding agents on US software engineering jobs
     ⬜ Research company adoption of AI coding agents and collect credible supporting evidence
🔧 task  {"description": "Research hiring trends and salary impact of AI coding agents on US software engineering jobs. Look for: 1) Recent…
🔧 task  {"description": "Research company adoption of AI coding agents and collect credible supporting evidence. Look for: 1) Company adop…
   ↳ task: **US Software Engineering Hiring & AI Coding Agent Salary Impact (2023-2026)** **Main Findings:** - **Hiring:** Junior SWE hiring declining (Pave: P1/…  (2,429 chars)
   ↳ task: **Main Findings** - **Adoption acceleration**: Jellyfish data (400+ US companies, Dec 2024–May 2025) shows agentic AI codin

#### Research workspace FileSystem

In [19]:
import re
import time

from deepagents import FilesystemPermission
from deepagents.backends import CompositeBackend, FilesystemBackend


def make_research_id(brief: str) -> str:
    """Readable, filesystem-safe and unique: first words of the brief + a timestamp."""
    words = re.sub(r"[^a-z0-9 ]+", "", brief.lower()).split()
    slug = "-".join(words[:6])[:40].strip("-") or "research"
    return f"{slug}-{time.strftime('%m%d-%H%M')}"


def start_run(brief: str, research_id: str | None = None) -> str:
    """Create this run's folder. The id comes from code (or the caller), never from the model."""
    research_id = research_id or make_research_id(brief)
    run = WORKSPACE / "research" / research_id
    for sub in ["sources", "findings", "claims", "reviews", "reports"]:
        (run / sub).mkdir(parents=True, exist_ok=True)
    (run / "request.md").write_text(f"# Research request\n\n{brief}\n")
    return research_id


def build_backend(research_id: str) -> CompositeBackend:
    """The agent's whole world. '/' is this run's folder; skills and memory are mounted beside it."""
    return CompositeBackend(
        default=FilesystemBackend(root_dir=WORKSPACE / "research" / research_id, virtual_mode=True),
        routes={
            "/skills/": FilesystemBackend(root_dir=WORKSPACE / "skills", virtual_mode=True),
            "/memory/": FilesystemBackend(root_dir=WORKSPACE / "memory", virtual_mode=True),
        },
    )


# Static rules: mounted skills and memory can be read but never changed.
PROTECTED = [FilesystemPermission(operations=["write"], paths=["/skills/**", "/memory/**"], mode="deny")]

print(make_research_id(BRIEF))

research-the-impact-of-ai-coding-1115-0734


In [20]:
RESEARCHER_PERMISSIONS = [
    FilesystemPermission(operations=["write"], paths=["/sources/**", "/findings/**"], mode="allow"),
    FilesystemPermission(operations=["write"], paths=["/**"], mode="deny"),
]


def file_rules(kind: str) -> str:
    return f"""
File rules (mandatory):
- Your task starts with a line "Slug: <slug>" (lowercase letters, digits, hyphens). The slug makes your file names
  unique, so you never overwrite another agent's work. If no slug is given, derive one from the topic and say so.
- Write exactly these two files with `write_file` BEFORE you reply, and nowhere else:
  - /sources/{kind}_<slug>.json
  - /findings/{kind}_<slug>.md
- The sources file is a JSON list of {{"title", "url", "publisher", "scope"}}
  where scope is "region-specific" or "global". Use exactly these keys.
- The findings file is bullet findings, each followed by its source URL.
- Your reply must be short: 5 bullets at most, limitations, source count, and the two file paths.
- If you could not find usable evidence, still write the files and say so inside them."""


web_spec, academic_spec = research_subagents
fs_subagents = [
    {**web_spec, "system_prompt": web_spec["system_prompt"] + file_rules("web"), "permissions": RESEARCHER_PERMISSIONS},
    {**academic_spec, "system_prompt": academic_spec["system_prompt"] + file_rules("academic"), "permissions": RESEARCHER_PERMISSIONS},
]

In [21]:
MANAGER_PROMPT = """You are the Research Manager of a research desk.
Your workspace root is `/`. The brief is also saved at /request.md. Folders: /sources, /findings, /claims, /reviews, /reports.

Follow these steps in order:
1. Write a todo list that splits the brief into independent research dimensions.
2. Write /plan.md (dimensions, who researches each, what evidence you need).
3. Delegate with the `task` tool to `web_researcher` and `academic_researcher`. Each task description MUST start
   with a line `Slug: <slug>`, a short unique name for that sub-topic (for example `functionality`, `cost`).
   Every slug must be unique across the whole run: parallel and follow-up tasks write files named after the slug,
   and a repeated slug would overwrite earlier work. After the slug line give the topic, the specific question and
   the scope (region, time period). Start independent delegations in the same step.
4. Run `ls /findings` and read EVERY findings file with `read_file`. Do not rely only on the specialists' chat replies.
5. Write /claims/claims_and_evidence.json: a JSON list of {"claim", "source_url", "source_title"} for every claim you plan to make.
6. Write /reports/draft.md as a readable research report in full prose paragraphs, not bullet lists
   (bullets are allowed only in the Sources list). Use only claims from claims_and_evidence.json. Structure:
   - `# Title`
   - `## Executive summary`: one paragraph that directly answers the brief.
   - one `## ` section per research dimension, 2-3 paragraphs each, explaining what the evidence shows with
     concrete numbers and inline [n] citations. Connect findings; do not just list them.
   - `## Evidence gaps and caveats`: what is missing, conflicting, estimated, or only global rather than local.
   - `## Sources`: numbered list with title, publisher and URL.
7. Keep the todo list complete: every `write_todos` call must contain ALL items, finished ones included; only change
   statuses. Mark an item `in_progress` when you start it and `completed` as soon as its step is done. Before your
   final answer every item must be `completed`.

Never do heavy searching yourself. The filesystem is the research workspace; the conversation is not the research database.
Always cite sources. Do not invent facts or URLs."""


def build_fs_agent(research_id: str):
    return create_deep_agent(
        name="research-manager",
        model=MODEL,
        middleware=[make_todo_middleware()],
        subagents=fs_subagents,
        backend=build_backend(research_id),
        permissions=PROTECTED,
        system_prompt=MANAGER_PROMPT,
    )

In [22]:
research_id = start_run(BRIEF)
print("Research id:", research_id)

fs_agent = build_fs_agent(research_id)   # its filesystem root is this run's folder
show_harness(fs_agent, "4 · + filesystem backend, permissions, file contract", subagents=fs_subagents)

result = fs_agent.invoke(
    {"messages": [{"role": "user", "content": f"Brief: {BRIEF}\nKeep it to two dimensions."}]},
    config=run_config("filesystem-agent", session_id=research_id),
)
show_trace(result, answer_chars=800)

print()
for p in sorted((WORKSPACE / "research" / research_id).rglob("*")):
    if p.is_file():
        print(p.relative_to(WORKSPACE), f"({p.stat().st_size} bytes)")

Research id: research-the-impact-of-ai-coding-1115-0737
🧩 HARNESS: 4 · + filesystem backend, permissions, file contract
Hook nodes (middleware that runs in the loop):
   • _ParentNotifierMiddleware.after_agent
   • PatchToolCallsMiddleware.before_agent
   • TodoListMiddleware.after_model
Tools (10): ls, read_file, write_file, edit_file, delete, glob, grep, execute, task, write_todos
Subagents:
   • web_researcher: tools=['search_web', 'extract_pages']  [2 permission rules]
   • academic_researcher: tools=['search_scholarly']  [2 permission rules]
➕ New since the previous harness shown: (nothing; the difference is in prompts, backend or config)


👤 USER: Brief: Research the impact of AI coding agents on software engineering jobs in US. Cover hiring trends, salary impact, company adoption, and credible supporting evidence. Keep it to two dimensions.

📝 write_todos
     ✅ Step 1: Write a todo list splitting the brief into two independent research dimensions.
     🔄 Step 2: Write /plan.md with dimensions, researchers, and evidence needed.
     ⬜ Step 3: Delegate research tasks to web_researcher and academic_researcher (slugs: hiring-trends-sala…
     ⬜ Step 4: Run ls /findings and read every findings file with read_file.
     ⬜ Step 5: Write /claims/claims_and_evidence.json with claim, source_url, source_title for each claim.
     ⬜ Step 6: Write /reports/draft.md as a readable research report in full prose with citations.
     ⬜ Step 7: Final review – mark all todos completed and deliver the draft report.
🔧 write_file  {"file_path": "/plan.md", "content": "# Research Plan: Impact of AI Coding Agents on Software Engineering Jobs i